# Embedding Evaluation

In [5]:
import numpy as np
import sys
import os
import pandas as pd

from pathlib import Path
sys.path.append(os.path.abspath('..'))

from src.skipgram import SkipGramModel,train_model
from src.negative_sampling import build_unigram_table
from src.tokenizer import Tokenizer

In [ ]:



with open("../data/corpus.txt", "r", encoding='utf-8') as f:
    raw_data = f.read()
    # Split the string by spaces and convert each "number" to an actual integer
    corpus_ids = [int(id_str) for id_str in raw_data.split()]
    
    
#tok.build_vocab(raw_data)


print(f"First 10 IDs: {corpus_ids[:10]}")

First 10 IDs: [29, 5, 2, 78, 1942, 45, 1064, 12, 101, 146]


In [7]:
df = pd.read_csv('./../data/IMDB Dataset.csv')


#print(df.head())


tok = Tokenizer()

reviews = df['review'].tolist()

# for review in df['review']:
#     reviews.append(review.replace('\n',''))
    
    
# print(reviews[:20])

tok.build_vocab(reviews)

In [3]:
unigram_table = build_unigram_table(corpus_ids,10000)

### Training

In [4]:
model = SkipGramModel(10000,100)

train_model(model,[corpus_ids[:50000]],unigram_table,10,5,0.025)

Epoch 1 | Pairs: 10000 | LR: 0.02500 | Loss: 4.15829
Epoch 1 | Pairs: 20000 | LR: 0.02500 | Loss: 4.15146
Epoch 1 | Pairs: 30000 | LR: 0.02500 | Loss: 4.11814
Epoch 1 | Pairs: 40000 | LR: 0.02500 | Loss: 4.06500
Epoch 1 | Pairs: 50000 | LR: 0.02500 | Loss: 4.00589
Epoch 1 | Pairs: 60000 | LR: 0.02500 | Loss: 3.92917
Epoch 1 | Pairs: 70000 | LR: 0.02500 | Loss: 3.85172
Epoch 1 | Pairs: 80000 | LR: 0.02500 | Loss: 3.79167
Epoch 1 | Pairs: 90000 | LR: 0.02500 | Loss: 3.73820
Epoch 1 | Pairs: 100000 | LR: 0.02500 | Loss: 3.68556
Epoch 1 | Pairs: 110000 | LR: 0.02500 | Loss: 3.63401
Epoch 1 | Pairs: 120000 | LR: 0.02500 | Loss: 3.58423
Epoch 1 | Pairs: 130000 | LR: 0.02500 | Loss: 3.54696
Epoch 1 | Pairs: 140000 | LR: 0.02500 | Loss: 3.50606
Epoch 1 | Pairs: 150000 | LR: 0.02500 | Loss: 3.47161
Epoch 1 | Pairs: 160000 | LR: 0.02500 | Loss: 3.43695
Epoch 1 | Pairs: 170000 | LR: 0.02500 | Loss: 3.40066
Epoch 1 | Pairs: 180000 | LR: 0.02500 | Loss: 3.36675
Epoch 1 | Pairs: 190000 | LR: 0.02500

In [12]:
def find_nearest_neighbors(query_word,word2idx,idx2word,normalized_matrix,top_k=5):
    if query_word not in word2idx:
        print(f"{query_word} is not in the dictionary")
        return
    
    
    #1 get the id and vector for the query word
    query_id = word2idx[query_word]
    query_vector = normalized_matrix[query_id]
    
    #2. cosine similarity 
    similarities = np.dot(normalized_matrix,query_vector)
    
    #3. sort  the indices based on similarity scores
    sorted_indices = np.argsort(similarities)
    
    #4. Grab the top k indices
    #remove the last index and reverse,
    best_indices = sorted_indices[-(top_k + 1):-1][::-1]
    
    print(f"words most similar to '{query_word}':")
    for idx in best_indices:
        print(f" {idx2word[idx]} (Score: {similarities[idx]:.4f})")
            

In [10]:
norms = np.linalg.norm(model.W1, axis=1,keepdims=True)

norms[norms == 0] = 1e-10

normalized_W1 = model.W1 / norms

idx2word = {idx: word for word, idx in tok.word_to_id.items()}

print(norms.shape)

(10000, 1)


In [13]:
find_nearest_neighbors("scary",tok.word_to_id,idx2word,normalized_W1)

words most similar to 'scary':
 phony (Score: 0.8895)
 disturbing (Score: 0.8806)
 chilling (Score: 0.8655)
 nudity (Score: 0.8611)
 follows (Score: 0.8575)


In [ ]:
def test_analogy(word_a,word_b,word_c,word2idx,idx2word,matrix,top_k=10):
    #Make sure all words are in our dictionary
    for w in [word_a,word_b,word_c]:
        if w not in word2idx:
            print(f"'{w}' is not in vocabulary")
            return
        
    #1. Get the vectors
    vec_a = matrix[word2idx[word_a]]
    vec_b = matrix[word2idx[word_b]]
    vec_c = matrix[word2idx[word_c]]
    print(matrix.shape)
    print(vec_a.shape)
    
    print(word2idx[word_a])
    
    
    #2. Do the math: target = A - B + C
    target_vector = vec_a - vec_b + vec_c
    
    #3. Calculate similarities across the whole matrix
    similarities = np.dot(matrix, target_vector)
    
    #4. Sort and slice the top K (including the #1 best match)
    sorted_indices =  np.argsort(similarities)
    best_indices = sorted_indices[-top_k:][::-1]
    
    
    print(f"Analogy: {word_a} - {word_b} + {word_c}")
    
    for idx in best_indices:
        if idx2word[idx] == word_a or idx2word[idx] == word_b or idx2word[idx] == word_c:
            continue
        print(f" {idx2word[idx]} (Score: {similarities[idx]:.4f})")

In [33]:
test_analogy("king", "man", "woman", tok.word_to_id, idx2word, normalized_W1)

(10000, 100)
(100,)
762
Analogy: king - man + woman
 king (Score: 0.9515)
 grabs (Score: 0.9217)
 woman (Score: 0.9206)
 thoroughly (Score: 0.9125)
 210 (Score: 0.9007)
 opposite (Score: 0.8982)
 edward (Score: 0.8966)
 george (Score: 0.8962)
 currently (Score: 0.8929)
 technical (Score: 0.8849)
